In [3]:
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from scipy.stats import mannwhitneyu
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests


PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "packages/network-security-ai/pyproject.toml").is_file())

TRAIN_V2_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "network_ml_split_v2"
    / "train.parquet"
)

TOP40_PATH = (
    PROJECT_ROOT
    / "results"
    / "feature_selection"
    / "top40_features.csv"
)

STAT_DIR = (
    PROJECT_ROOT
    / "results"
    / "statistical_analysis_v4"
)

STAT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEMP_DIR = (
    PROJECT_ROOT
    / "data"
    / "tmp"
    / "duckdb_stat_v4"
)

TEMP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


top40_df = pd.read_csv(
    TOP40_PATH
)

TOP40_FEATURES = (
    top40_df["feature"]
    .astype(str)
    .tolist()
)

assert len(TOP40_FEATURES) == 40

print("Top40:", len(TOP40_FEATURES))
print("Train:", TRAIN_V2_PATH)
print("Output:", STAT_DIR)

Top40: 40
Train: /home/geonug/kdt-linux/ML/data/processed/network_ml_split_v2/train.parquet
Output: /home/geonug/kdt-linux/ML/results/statistical_analysis_v4


In [4]:
con = duckdb.connect()

con.execute(
    "SET memory_limit='2GB'"
)

con.execute(
    "SET threads=2"
)

con.execute(
    f"SET temp_directory='{TEMP_DIR}'"
)

con.execute(
    "SET preserve_insertion_order=false"
)


def quote_identifier(name):
    return '"' + name.replace('"', '""') + '"'


TOP40_SQL = ", ".join(
    quote_identifier(f)
    for f in TOP40_FEATURES
)

In [5]:
STAT_SAMPLE_PATH = (
    STAT_DIR
    / "benign_attack_sample.parquet"
)

SAMPLE_PER_GROUP = 100_000

In [6]:
# ============================================================
# 안전한 Benign / Attack 통계 샘플 생성
# - global ORDER BY 없음
# - 40개 feature hash 없음
# - WSL 메모리 보호
# ============================================================

STAT_SAMPLE_PATH = (
    STAT_DIR
    / "benign_attack_sample.parquet"
)

SAMPLE_PER_GROUP = 100_000


# 이전 실행 중 깨진 파일이 남았는지 확인
if STAT_SAMPLE_PATH.exists():
    try:
        check = con.execute(
            f"""
            SELECT COUNT(*)
            FROM read_parquet('{STAT_SAMPLE_PATH}')
            """
        ).fetchone()[0]

        if check != SAMPLE_PER_GROUP * 2:
            print(
                "[WARNING] incomplete sample:",
                check
            )

            STAT_SAMPLE_PATH.unlink()

    except Exception:
        print(
            "[WARNING] broken parquet removed"
        )

        STAT_SAMPLE_PATH.unlink()


# ============================================================
# 생성
# ============================================================

if not STAT_SAMPLE_PATH.exists():

    print("[START] statistical sample")

    con.execute(
        f"""
        COPY (

            (
                SELECT
                    {TOP40_SQL},
                    label_id,
                    0::TINYINT AS binary_label

                FROM read_parquet(
                    '{TRAIN_V2_PATH}'
                )

                WHERE label_id = 0

                LIMIT {SAMPLE_PER_GROUP}
            )

            UNION ALL

            (
                SELECT
                    {TOP40_SQL},
                    label_id,
                    1::TINYINT AS binary_label

                FROM read_parquet(
                    '{TRAIN_V2_PATH}'
                )

                WHERE label_id != 0

                LIMIT {SAMPLE_PER_GROUP}
            )

        )

        TO '{STAT_SAMPLE_PATH}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        "[DONE]",
        STAT_SAMPLE_PATH
    )

else:

    print(
        "[SKIP]",
        STAT_SAMPLE_PATH
    )

[WARNING] broken parquet removed
[START] statistical sample
[DONE] /home/geonug/kdt-linux/ML/results/statistical_analysis_v4/benign_attack_sample.parquet


In [7]:
check_df = con.execute(
    f"""
    SELECT
        binary_label,
        COUNT(*) AS rows,
        COUNT(DISTINCT label_id) AS classes

    FROM read_parquet(
        '{STAT_SAMPLE_PATH}'
    )

    GROUP BY binary_label
    ORDER BY binary_label
    """
).df()


print(
    check_df.to_string(
        index=False
    )
)

 binary_label   rows  classes
            0 100000        1
            1 100000        1


In [8]:
sample_check = con.execute(
    f"""
    SELECT
        binary_label,
        COUNT(*) AS rows
    FROM read_parquet(
        '{STAT_SAMPLE_PATH}'
    )
    GROUP BY binary_label
    ORDER BY binary_label
    """
).df()

print(
    sample_check.to_string(
        index=False
    )
)

 binary_label   rows
            0 100000
            1 100000


In [9]:
stat_df = pd.read_parquet(
    STAT_SAMPLE_PATH
)

print(
    "Shape:",
    stat_df.shape
)

print(
    "Memory:",
    f"{stat_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print(
    stat_df[
        "binary_label"
    ].value_counts()
)

Shape: (200000, 42)
Memory: 61.61 MB
binary_label
0    100000
1    100000
Name: count, dtype: int64


In [10]:
def mann_whitney_analysis(
    benign,
    attack,
):

    benign = np.asarray(
        benign,
        dtype=np.float64,
    )

    attack = np.asarray(
        attack,
        dtype=np.float64,
    )


    benign = benign[
        np.isfinite(benign)
    ]

    attack = attack[
        np.isfinite(attack)
    ]


    n_benign = len(benign)
    n_attack = len(attack)


    if (
        n_benign == 0
        or n_attack == 0
    ):

        return {
            "u_statistic": np.nan,
            "p_value": np.nan,
            "effect_size": np.nan,
            "n_benign": n_benign,
            "n_attack": n_attack,
        }


    result = mannwhitneyu(
        benign,
        attack,
        alternative="two-sided",
        method="asymptotic",
    )


    u = float(
        result.statistic
    )


    # -1 ~ +1
    # 양수: Benign 값이 상대적으로 큰 경향
    # 음수: Attack 값이 상대적으로 큰 경향
    effect_size = (
        2.0
        * u
        / (
            n_benign
            * n_attack
        )
        - 1.0
    )


    return {
        "u_statistic": u,
        "p_value": float(
            result.pvalue
        ),
        "effect_size": effect_size,
        "n_benign": n_benign,
        "n_attack": n_attack,
    }

In [11]:
results = []


benign_mask = (
    stat_df["binary_label"]
    == 0
)

attack_mask = (
    stat_df["binary_label"]
    == 1
)


for rank, feature in enumerate(
    TOP40_FEATURES,
    start=1,
):

    result = mann_whitney_analysis(
        stat_df.loc[
            benign_mask,
            feature,
        ],
        stat_df.loc[
            attack_mask,
            feature,
        ],
    )


    results.append({
        "rank_top40": rank,
        "feature": feature,
        **result,
    })


mw_df = pd.DataFrame(
    results
)

print(
    mw_df.head().to_string(
        index=False
    )
)

 rank_top40           feature  u_statistic  p_value  effect_size  n_benign  n_attack
          1 Fwd Act Data Pkts 6491635609.0      0.0     0.298327    100000    100000
          2  Fwd Seg Size Min 3780125603.0      0.0    -0.243975    100000    100000
          3    Fwd Header Len 5819243753.0      0.0     0.163849    100000    100000
          4      ECE Flag Cnt 3794200000.0      0.0    -0.241160    100000    100000
          5 Init Fwd Win Byts 4508209711.0      0.0    -0.098358    100000    100000


In [12]:
valid_mask = (
    mw_df["p_value"]
    .notna()
)


reject, p_fdr, _, _ = multipletests(
    mw_df.loc[
        valid_mask,
        "p_value",
    ],
    alpha=0.05,
    method="fdr_bh",
)


mw_df.loc[
    valid_mask,
    "p_value_fdr"
] = p_fdr


mw_df.loc[
    valid_mask,
    "significant_fdr_005"
] = reject

In [13]:
mw_df[
    "abs_effect_size"
] = (
    mw_df[
        "effect_size"
    ].abs()
)

In [14]:
def effect_level(value):

    value = abs(value)

    if value >= 0.5:
        return "large"

    if value >= 0.3:
        return "medium"

    if value >= 0.1:
        return "small"

    return "negligible"


mw_df[
    "effect_level"
] = (
    mw_df[
        "effect_size"
    ]
    .apply(
        effect_level
    )
)

In [15]:
mw_ranked = (
    mw_df
    .sort_values(
        [
            "abs_effect_size",
            "p_value_fdr",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    mw_ranked[
        [
            "feature",
            "p_value",
            "p_value_fdr",
            "effect_size",
            "abs_effect_size",
            "effect_level",
            "significant_fdr_005",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)

          feature      p_value  p_value_fdr  effect_size  abs_effect_size effect_level significant_fdr_005
         Dst Port            0            0     -0.73221          0.73221        large                True
       Fwd Pkts/s            0            0    -0.551948         0.551948        large                True
    Flow Duration            0            0     0.499302         0.499302       medium                True
    Flow IAT Mean            0            0     0.492709         0.492709       medium                True
     Pkt Size Avg            0            0     0.490014         0.490014       medium                True
     Flow IAT Max            0            0     0.483977         0.483977       medium                True
  TotLen Bwd Pkts            0            0     0.418783         0.418783       medium                True
 Subflow Bwd Byts            0            0     0.418783         0.418783       medium                True
     Pkt Len Mean            0       

In [16]:
MW_PATH = (
    STAT_DIR
    / "mann_whitney_top40.csv"
)

mw_ranked.to_csv(
    MW_PATH,
    index=False,
)

print(
    "Saved:",
    MW_PATH
)

Saved: /home/geonug/kdt-linux/ML/results/statistical_analysis_v4/mann_whitney_top40.csv


In [17]:
print(
    "FDR significant:",
    int(
        mw_ranked[
            "significant_fdr_005"
        ].sum()
    ),
    "/",
    len(mw_ranked),
)


print(
    "\nEffect size distribution:"
)

print(
    mw_ranked[
        "effect_level"
    ].value_counts()
)


print(
    "\n=== TOP 15 BY EFFECT SIZE ==="
)

print(
    mw_ranked[
        [
            "feature",
            "p_value_fdr",
            "effect_size",
            "effect_level",
        ]
    ]
    .head(15)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)

FDR significant: 38 / 40

Effect size distribution:
effect_level
small         17
medium        13
negligible     8
large          2
Name: count, dtype: int64

=== TOP 15 BY EFFECT SIZE ===
         feature  p_value_fdr  effect_size effect_level
        Dst Port            0     -0.73221        large
      Fwd Pkts/s            0    -0.551948        large
   Flow Duration            0     0.499302       medium
   Flow IAT Mean            0     0.492709       medium
    Pkt Size Avg            0     0.490014       medium
    Flow IAT Max            0     0.483977       medium
 TotLen Bwd Pkts            0     0.418783       medium
Subflow Bwd Byts            0     0.418783       medium
    Pkt Len Mean            0     0.416574       medium
 Bwd Pkt Len Max            0     0.409778       medium
     Fwd IAT Tot            0     0.369033       medium
     Fwd IAT Max            0     0.363362       medium
     Fwd IAT Std            0     0.354677       medium
  Bwd Header Len          

In [18]:
# ============================================================
# Kruskal-Wallis용 15-class sample
# - Train V2만 사용
# - 클래스별 최대 20,000 rows
# - global ORDER BY / random sort 없음
# ============================================================

KW_SAMPLE_DIR = (
    STAT_DIR
    / "kruskal_class_parts"
)

KW_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MAX_PER_CLASS = 20_000
N_CLASSES = 15


for label_id in range(N_CLASSES):

    output_path = (
        KW_SAMPLE_DIR
        / f"class_{label_id:02d}.parquet"
    )

    if output_path.exists():
        print(
            f"[SKIP] class={label_id}"
        )
        continue

    print(
        f"[START] class={label_id}"
    )

    con.execute(
        f"""
        COPY (
            SELECT
                {TOP40_SQL},
                label_id

            FROM read_parquet(
                '{TRAIN_V2_PATH}'
            )

            WHERE label_id = {label_id}

            LIMIT {MAX_PER_CLASS}
        )

        TO '{output_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )

    print(
        f"[DONE] class={label_id}"
    )

[START] class=0
[DONE] class=0
[START] class=1
[DONE] class=1
[START] class=2
[DONE] class=2
[START] class=3
[DONE] class=3
[START] class=4
[DONE] class=4
[START] class=5
[DONE] class=5
[START] class=6
[DONE] class=6
[START] class=7
[DONE] class=7
[START] class=8
[DONE] class=8
[START] class=9
[DONE] class=9
[START] class=10
[DONE] class=10
[START] class=11
[DONE] class=11
[START] class=12
[DONE] class=12
[START] class=13
[DONE] class=13
[START] class=14
[DONE] class=14


In [19]:
sample_summary = []

for label_id in range(N_CLASSES):

    path = (
        KW_SAMPLE_DIR
        / f"class_{label_id:02d}.parquet"
    )

    rows = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet('{path}')
        """
    ).fetchone()[0]

    sample_summary.append({
        "label_id": label_id,
        "rows": rows,
    })


sample_summary_df = pd.DataFrame(
    sample_summary
)

print(
    sample_summary_df.to_string(
        index=False
    )
)

print(
    "\nTotal:",
    sample_summary_df["rows"].sum()
)

 label_id  rows
        0 20000
        1 20000
        2   428
        3   166
        4 20000
        5  1203
        6 20000
        7 20000
        8 20000
        9 20000
       10  7766
       11 20000
       12 20000
       13    55
       14 20000

Total: 209618


In [20]:
class_dfs = {}

for label_id in range(N_CLASSES):

    path = (
        KW_SAMPLE_DIR
        / f"class_{label_id:02d}.parquet"
    )

    df = pd.read_parquet(
        path
    )

    class_dfs[label_id] = df

    print(
        f"class={label_id:2d}",
        f"rows={len(df):,}",
    )

class= 0 rows=20,000
class= 1 rows=20,000
class= 2 rows=428
class= 3 rows=166
class= 4 rows=20,000
class= 5 rows=1,203
class= 6 rows=20,000
class= 7 rows=20,000
class= 8 rows=20,000
class= 9 rows=20,000
class=10 rows=7,766
class=11 rows=20,000
class=12 rows=20,000
class=13 rows=55
class=14 rows=20,000


In [21]:
from scipy.stats import kruskal


def kruskal_feature_analysis(
    feature,
    class_dfs,
):

    groups = []

    group_sizes = []

    for label_id in sorted(
        class_dfs.keys()
    ):

        values = (
            class_dfs[label_id][feature]
            .to_numpy(
                dtype=np.float64,
                copy=False,
            )
        )

        values = values[
            np.isfinite(values)
        ]

        if len(values) > 0:
            groups.append(values)
            group_sizes.append(
                len(values)
            )


    k = len(groups)

    N = sum(
        group_sizes
    )


    if k < 2:

        return {
            "h_statistic": np.nan,
            "p_value": np.nan,
            "epsilon_squared": np.nan,
            "n_total": N,
            "n_groups": k,
        }


    # 모든 관측값이 완전히 동일하면
    # scipy가 오류를 낼 수 있으므로 별도 처리
    all_min = min(
        np.min(g)
        for g in groups
    )

    all_max = max(
        np.max(g)
        for g in groups
    )

    if all_min == all_max:

        return {
            "h_statistic": 0.0,
            "p_value": 1.0,
            "epsilon_squared": 0.0,
            "n_total": N,
            "n_groups": k,
        }


    result = kruskal(
        *groups,
        nan_policy="omit",
    )

    H = float(
        result.statistic
    )

    p = float(
        result.pvalue
    )


    epsilon_squared = (
        (H - k + 1)
        / (N - k)
    )

    # 이론적으로 음수가 나올 수 있으므로 0으로 제한
    epsilon_squared = max(
        0.0,
        float(epsilon_squared),
    )


    return {
        "h_statistic": H,
        "p_value": p,
        "epsilon_squared": epsilon_squared,
        "n_total": N,
        "n_groups": k,
    }

In [22]:
kw_results = []


for rank, feature in enumerate(
    TOP40_FEATURES,
    start=1,
):

    result = kruskal_feature_analysis(
        feature,
        class_dfs,
    )

    kw_results.append({
        "rank_top40": rank,
        "feature": feature,
        **result,
    })


kw_df = pd.DataFrame(
    kw_results
)


print(
    kw_df.head().to_string(
        index=False
    )
)

 rank_top40           feature   h_statistic  p_value  epsilon_squared  n_total  n_groups
          1 Fwd Act Data Pkts  75931.286310      0.0         0.362196   209618        15
          2  Fwd Seg Size Min 194581.044741      0.0         0.928265   209618        15
          3    Fwd Header Len  77544.133356      0.0         0.369890   209618        15
          4      ECE Flag Cnt 116223.874306      0.0         0.554428   209618        15
          5 Init Fwd Win Byts 125011.640189      0.0         0.596354   209618        15


In [23]:
valid_mask = (
    kw_df["p_value"]
    .notna()
)


reject, p_fdr, _, _ = multipletests(
    kw_df.loc[
        valid_mask,
        "p_value",
    ],
    alpha=0.05,
    method="fdr_bh",
)


kw_df.loc[
    valid_mask,
    "p_value_fdr"
] = p_fdr


kw_df.loc[
    valid_mask,
    "significant_fdr_005"
] = reject

In [24]:
def epsilon_effect_level(
    value
):

    if pd.isna(value):
        return "unknown"

    if value >= 0.14:
        return "large"

    if value >= 0.06:
        return "medium"

    if value >= 0.01:
        return "small"

    return "negligible"


kw_df[
    "effect_level"
] = (
    kw_df[
        "epsilon_squared"
    ]
    .apply(
        epsilon_effect_level
    )
)

In [25]:
kw_ranked = (
    kw_df
    .sort_values(
        [
            "epsilon_squared",
            "p_value_fdr",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


KW_RESULT_PATH = (
    STAT_DIR
    / "kruskal_wallis_top40.csv"
)


kw_ranked.to_csv(
    KW_RESULT_PATH,
    index=False,
)


print(
    "Saved:",
    KW_RESULT_PATH
)

Saved: /home/geonug/kdt-linux/ML/results/statistical_analysis_v4/kruskal_wallis_top40.csv


In [26]:
print(
    "FDR significant:",
    int(
        kw_ranked[
            "significant_fdr_005"
        ].sum()
    ),
    "/",
    len(kw_ranked),
)


print(
    "\nEffect size distribution:"
)

print(
    kw_ranked[
        "effect_level"
    ].value_counts()
)


print(
    "\n=== TOP 20 KRUSKAL-WALLIS ==="
)

print(
    kw_ranked[
        [
            "feature",
            "h_statistic",
            "p_value_fdr",
            "epsilon_squared",
            "effect_level",
        ]
    ]
    .head(20)
    .to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)

FDR significant: 40 / 40

Effect size distribution:
effect_level
large         38
small          1
negligible     1
Name: count, dtype: int64

=== TOP 20 KRUSKAL-WALLIS ===
          feature  h_statistic  p_value_fdr  epsilon_squared effect_level
 Fwd Seg Size Min       194581            0         0.928265        large
         Dst Port       172299            0         0.821957        large
    Flow IAT Mean       153586            0         0.732682        large
       Fwd Pkts/s       152647            0         0.728202        large
     Flow IAT Max       151961            0         0.724929        large
    Flow Duration       148740            0         0.709561        large
Init Fwd Win Byts       125012            0         0.596354        large
      Fwd IAT Max       122040            0         0.582177        large
      Fwd IAT Tot       121982            0         0.581902        large
       Bwd Pkts/s       119392            0         0.569544        large
     ECE Flag

In [27]:
print(
    kw_ranked[
        kw_ranked[
            "feature"
        ].isin([
            "PSH Flag Cnt",
            "FIN Flag Cnt",
            "Fwd URG Flags",
            "Dst Port",
            "Fwd Pkts/s",
        ])
    ][
        [
            "feature",
            "p_value_fdr",
            "epsilon_squared",
            "effect_level",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)

      feature  p_value_fdr  epsilon_squared effect_level
     Dst Port            0         0.821957        large
   Fwd Pkts/s            0         0.728202        large
 PSH Flag Cnt            0         0.211613        large
Fwd URG Flags            0        0.0103481        small
 FIN Flag Cnt   6.895e-250       0.00571159   negligible


In [28]:
# ============================================================
# Top40 Statistical Validation 통합
# ============================================================

statistical_validation = (
    mw_ranked[
        [
            "rank_top40",
            "feature",
            "p_value_fdr",
            "effect_size",
            "abs_effect_size",
            "effect_level",
        ]
    ]
    .rename(
        columns={
            "p_value_fdr": "mw_fdr_p",
            "effect_size": "mw_effect",
            "abs_effect_size": "mw_abs_effect",
            "effect_level": "mw_effect_level",
        }
    )
    .merge(
        kw_ranked[
            [
                "feature",
                "p_value_fdr",
                "epsilon_squared",
                "effect_level",
            ]
        ].rename(
            columns={
                "p_value_fdr": "kw_fdr_p",
                "epsilon_squared": "kw_epsilon_squared",
                "effect_level": "kw_effect_level",
            }
        ),
        on="feature",
        how="left",
    )
    .sort_values("rank_top40")
)


OUTPUT_PATH = (
    STAT_DIR
    / "top40_statistical_validation.csv"
)

statistical_validation.to_csv(
    OUTPUT_PATH,
    index=False,
)


print(
    statistical_validation.to_string(
        index=False,
        float_format=lambda x: f"{x:.6g}",
    )
)

print("\nSaved:", OUTPUT_PATH)

 rank_top40           feature     mw_fdr_p  mw_effect  mw_abs_effect mw_effect_level   kw_fdr_p  kw_epsilon_squared kw_effect_level
          1 Fwd Act Data Pkts            0   0.298327       0.298327           small          0            0.362196           large
          2  Fwd Seg Size Min            0  -0.243975       0.243975           small          0            0.928265           large
          3    Fwd Header Len            0   0.163849       0.163849           small          0             0.36989           large
          4      ECE Flag Cnt            0   -0.24116        0.24116           small          0            0.554428           large
          5 Init Fwd Win Byts            0 -0.0983581      0.0983581      negligible          0            0.596354           large
          6      URG Flag Cnt            0    0.03189        0.03189      negligible          0            0.214842           large
          7      RST Flag Cnt            0   -0.24117        0.24117        